In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
!pip install openpyxl

In [ ]:
%cd ..

In [ ]:
import os
os.chdir(r'C:\Users\v\Desktop\ECGPerturb-main')
print("Dossier actuel :", os.getcwd())

In [ ]:
# Utilise ! au lieu de %run pour isoler les workers de l'interface Jupyter
!python ecg_generator/ecg_main.py --repeats 1000 --generation-workers 8

In [ ]:
!python DataAugmentation/pipeline.py --augmentation-workers 8


In [ ]:
!python DataAugmentation/pipeline.py --resume --augmentation-workers 8


In [ ]:
# Visualisation NPZ : image augmentee | points d'intersection (grille major) | superposition
import os, sys, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
sys.path.insert(0, r'C:\Users\v\Desktop\ECGPerturb-main')
from shared.npz_schema import load_unified_npz

%matplotlib inline

# === A modifier ===
NPZ_PATH      = r'data/output_augmentation/npz/ECG_032_10_p0_aug.npz'
IMG_PATH      = r'data/output_augmentation/images/ECG_032_10_p0_aug.webp'
POINT_RADIUS  = 3            # rayon des points (px)
OVERLAY_COLOR = (255, 0, 0)  # rouge

# === Chargement ===
data = load_unified_npz(NPZ_PATH)
img  = np.array(Image.open(IMG_PATH).convert('RGB'))
H, W = img.shape[:2]

pts = data['grid_major_5mm']  # Nx2 en coordonnees image augmentee
print(f'{len(pts)} intersections grille major | image {W}x{H}')

# === Masque points blancs sur fond noir ===
mask = np.zeros((H, W), dtype=np.uint8)
xs = np.round(pts[:, 0]).astype(int)
ys = np.round(pts[:, 1]).astype(int)
valid = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
xs, ys = xs[valid], ys[valid]
for dx in range(-POINT_RADIUS, POINT_RADIUS + 1):
    for dy in range(-POINT_RADIUS, POINT_RADIUS + 1):
        if dx*dx + dy*dy <= POINT_RADIUS*POINT_RADIUS:
            mask[np.clip(ys + dy, 0, H - 1), np.clip(xs + dx, 0, W - 1)] = 255

# === Superposition ===
overlay = img.copy()
overlay[mask > 0] = OVERLAY_COLOR

# === Affichage ===
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
axes[0].imshow(img);                                 axes[0].set_title('Image augmentee');                   axes[0].axis('off')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255); axes[1].set_title(f'Points intersection ({len(xs)})');  axes[1].axis('off')
axes[2].imshow(overlay);                             axes[2].set_title('Superposition');                     axes[2].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Visualisation PNG mask grille major : image augmentee | mask | superposition
import os, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

# === A modifier ===
STEM          = 'ECG_033_05_p0_aug'
OVERLAY_COLOR = (255, 0, 0)
OVERLAY_ALPHA = 0.6

IMG_PATH  = f'data/output_augmentation/images/{STEM}.webp'
MASK_PATH = f'data/output_augmentation/masks/{STEM}/mask_grid_major.png'

# === Chargement ===
img  = np.array(Image.open(IMG_PATH).convert('RGB'))
mask = np.array(Image.open(MASK_PATH).convert('L'))
H, W = img.shape[:2]
print(f'Image {W}x{H} | Mask {mask.shape[1]}x{mask.shape[0]}')

# === Superposition (mask en couleur, semi-transparent) ===
overlay = img.copy().astype(np.float32)
mask_bool = mask > 127
color = np.array(OVERLAY_COLOR, dtype=np.float32)
overlay[mask_bool] = (1 - OVERLAY_ALPHA) * overlay[mask_bool] + OVERLAY_ALPHA * color
overlay = overlay.clip(0, 255).astype(np.uint8)

# === Affichage ===
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
axes[0].imshow(img);                                axes[0].set_title('Image augmentee');         axes[0].axis('off')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255); axes[1].set_title('Mask grille major (PNG)'); axes[1].axis('off')
axes[2].imshow(overlay);                            axes[2].set_title('Superposition');           axes[2].axis('off')
plt.tight_layout(); plt.show()
